# Cuaderno B: recompensa del Ejecutor corregida (semillas 3000 y 4000, opción A 2000)

**Qué se corrige:** en ABIDES la recompensa comparaba el precio pagado con el precio medio de 30 segundos *después*, que ya incluye el alza que provoca la propia compra. Así una compra cara podía salir premiada, y la recompensa casi no se relacionaba con el costo (correlación 0,04 a 0,22). Ahora se compara con el precio que vio la red **al decidir**, como en Poisson. Detalle en `docs/revision_opus_PS.md`.

**Cómo usarlo:** ejecuta las celdas **en orden** (o "Ejecutar todas"; si la celda 1 corta la ejecución, vuelve a lanzar "Ejecutar todas"). Dura unas **2 horas y 45 minutos**. Si se corta, vuelve a ejecutar todo: retoma donde quedó.

Todo se guarda en Drive en **`slippage_colab/resultados/rmid_B`** (carpeta propia; el otro cuaderno usa otra) y al final se arma `rmid_B.zip`. Los dos cuadernos (A y B) se pueden correr al mismo tiempo.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama con el parche de R_E; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q -f origin/feature/ps-revision-opus
!cd /content/slippage && git log --oneline -1 && grep -c "mid_decision" scripts/colab/entrenamiento_inicial_ps.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Paso 1: verificación del parche (≈ 5 minutos, no entrena)

Comprueba que la recompensa nueva usa el precio que vio la red al decidir y que se relaciona con el costo. Al final imprime **OK** o **PARAR**. Si dice **PARAR**, no sigas: avísale a PS y manda el archivo `rmid_B/verificacion_mid_decision.json`.

In [ ]:
# Verificación: misma política aleatoria en 40 semillas, con el parche apagado y encendido
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/verificar_mid_decision_ps.py --out {RESULTS_DIR}/rmid_B/verificacion_mid_decision.json 2>&1 | grep -v UserWarning


In [ ]:
# Corta aquí si la verificación no dio OK (así no se gastan horas de entrenamiento)
import json
v = json.load(open(f'{RESULTS_DIR}/rmid_B/verificacion_mid_decision.json', encoding='utf-8'))
print(v['veredicto'])
assert v['veredicto'].startswith('OK'), 'PARAR: avisa a PS antes de seguir'


## Paso 2: entrenamientos (≈ 50 minutos cada uno)

Recompensa original del Título I **medida contra el precio de la decisión**, con penalización de 30 bps por acción sin comprar y γ = 1. Las dos primeras usan la opción 0 de D3 (riesgo sobre lo ejecutado, la del Título I) y la última la opción A (riesgo sobre lo pendiente), para que la decisión D3 se tome con la recompensa corregida.

In [ ]:
# Opción 0, semilla de red 3000 (≈ 50 min)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/entrenamiento_inicial_ps.py --mid-decision --gamma 1.0 --penalizacion-bps 30 --seed 3000 --episodes 1000 --out-dir {RESULTS_DIR}/rmid_B/d3_opcion0_s3000 --max-minutes 150 2>&1 | grep -v UserWarning


In [ ]:
# Opción 0, semilla de red 4000 (≈ 50 min)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/entrenamiento_inicial_ps.py --mid-decision --gamma 1.0 --penalizacion-bps 30 --seed 4000 --episodes 1000 --out-dir {RESULTS_DIR}/rmid_B/d3_opcion0_s4000 --max-minutes 150 2>&1 | grep -v UserWarning


In [ ]:
# Opción A, semilla de red 2000 (≈ 50 min)
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/entrenamiento_inicial_ps.py --mid-decision --gamma 1.0 --penalizacion-bps 30 --riesgo-sobre pendiente --seed 2000 --episodes 1000 --out-dir {RESULTS_DIR}/rmid_B/d3_opcionA_s2000 --max-minutes 150 2>&1 | grep -v UserWarning


## Paso 3: zip para descargar

In [ ]:
# CELDA FINAL: arma un zip pequeño (sin checkpoints) para pasárselo a PS
import os, zipfile
base = f'{RESULTS_DIR}/rmid_B'
archivos = [os.path.join(c, a) for c, _, fs in os.walk(base) if 'checkpoints_' not in c for a in fs if a.endswith(('.json', '.csv'))]
print(len(archivos), 'archivos')
zpath = f'{RESULTS_DIR}/rmid_B.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print('zip listo:', zpath, '-> descárgalo desde slippage_colab/resultados en Drive')
